In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
import os
from sklearn.preprocessing import StandardScaler

print("Librerías importadas correctamente")

## Carga de la carpeta


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Dimensiones y primeras 5 lineas


In [ ]:
columnas = [
    'timestamp', 'activityID', 'heart_rate',
    'hand_temp','hand_acc16_1','hand_acc16_2','hand_acc16_3',
    'hand_acc6_1','hand_acc6_2','hand_acc6_3',
    'hand_gyr1','hand_gyr2','hand_gyr3',
    'hand_mag1','hand_mag2','hand_mag3',
    'hand_or1','hand_or2','hand_or3','hand_or4',
    'chest_temp','chest_acc16_1','chest_acc16_2','chest_acc16_3',
    'chest_acc6_1','chest_acc6_2','chest_acc6_3',
    'chest_gyr1','chest_gyr2','chest_gyr3',
    'chest_mag1','chest_mag2','chest_mag3',
    'chest_or1','chest_or2','chest_or3','chest_or4',
    'ankle_temp','ankle_acc16_1','ankle_acc16_2','ankle_acc16_3',
    'ankle_acc6_1','ankle_acc6_2','ankle_acc6_3',
    'ankle_gyr1','ankle_gyr2','ankle_gyr3',
    'ankle_mag1','ankle_mag2','ankle_mag3',
    'ankle_or1','ankle_or2','ankle_or3','ankle_or4'
]

protocol_path = '/content/drive/MyDrive/pamap2+physical+activity+monitoring/PAMAP2_Dataset/Protocol/'
archivos = sorted(glob(protocol_path + '*.dat'))

print(f"Archivos encontrados: {archivos}") # Added for debugging

dfs = []
for archivo in archivos:
    df_temp = pd.read_csv(archivo, sep=' ', header=None, names=columnas)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)
print(f"Dataset cargado: {len(archivos)} sujetos")

In [ ]:
print("Dimensiones:")
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print("Primeras 5 filas: ")
# pd.set_option('display.max_columns', None)
df.head()


In [ ]:
# Ver NaN por columna
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

resumen = pd.DataFrame({'NaN count': missing, '% faltante': missing_pct})
print(resumen[resumen['NaN count'] > 0])

## 1. Creacion de subject_id, eliminacion de activityID=0 y cuaterniones


In [ ]:
df_raw_shape = df.shape

# 1. CREAR subject_id ANTES de cualquier operacion
# Detectamos reinicios del timestamp: cuando el tiempo baja, es un nuevo sujeto
df['subject_id'] = (df['timestamp'].diff() < 0).cumsum() + 1
n_sujetos = df['subject_id'].nunique()
print(f"Sujetos detectados automaticamente: {n_sujetos}")
print(f"Filas por sujeto:")
print(df['subject_id'].value_counts().sort_index().to_string())

# 2. Eliminar activityID = 0 (periodos transitorios sin actividad definida)
df = df[df['activityID'] != 0]
print(f"\nFilas tras eliminar activityID=0: {df.shape[0]:,}")

# 3. Eliminar columnas de orientacion (cuaterniones invalidos segun documentacion oficial)
orientation_cols = [col for col in df.columns if 'or' in col]
df = df.drop(columns=orientation_cols)
print(f"Columnas de cuaterniones eliminadas: {orientation_cols}")


## 2. Analisis de saturacion de sensores +/-6g vs +/-16g (datos crudos)

> **Por que aqui?** Este analisis debe hacerse sobre los datos en escala fisica real (m/s2),
> antes de interpolar y antes del Z-Score. Si lo hicieramos despues, perderiamos
> la referencia fisica de los limites del sensor y no podriamos detectar saturacion real.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ANÁLISIS DE SATURACIÓN ±6g vs ±16g + REEMPLAZO SELECTIVO POR ACTIVIDAD
# Ejecutar ANTES de interpolación y ANTES de Z-Score (datos en m/s²)
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ── Constantes físicas ─────────────────────────────────────────────────────────
LIMITE_6G  = 58.86    # 6  × 9.81 m/s²
LIMITE_16G = 156.96   # 16 × 9.81 m/s²
UMBRAL_SAT = 0.90     # Consideramos "en riesgo" si supera el 90% del límite
UMBRAL_PCT_DECISION = 10.0  # Si > 10% de muestras saturan en una actividad → reemplazar

ACTIVIDADES = {
    1: 'Lying', 2: 'Sitting', 3: 'Standing', 4: 'Walking',
    5: 'Running', 6: 'Cycling', 7: 'Nordic Walking',
    12: 'Ascending Stairs', 13: 'Descending Stairs',
    16: 'Vacuum Cleaning', 17: 'Ironing', 24: 'Rope Jumping'
}
INTENSIDAD = {
    5: 'alta', 24: 'alta', 12: 'alta', 13: 'alta',
    4: 'media', 7: 'media', 6: 'media', 16: 'media',
    1: 'baja', 2: 'baja', 3: 'baja', 17: 'baja'
}
COLOR_INT = {'alta': '#e63946', 'media': '#f4a261', 'baja': '#2a9d8f'}

# Sensores anatómicos: cada ubicación tiene sus propios ejes acc6 y acc16
SENSORES = {
    'hand':  {'acc6': ['hand_acc6_1', 'hand_acc6_2', 'hand_acc6_3'],
              'acc16': ['hand_acc16_1', 'hand_acc16_2', 'hand_acc16_3']},
    'chest': {'acc6': ['chest_acc6_1', 'chest_acc6_2', 'chest_acc6_3'],
              'acc16': ['chest_acc16_1', 'chest_acc16_2', 'chest_acc16_3']},
    'ankle': {'acc6': ['ankle_acc6_1', 'ankle_acc6_2', 'ankle_acc6_3'],
              'acc16': ['ankle_acc16_1', 'ankle_acc16_2', 'ankle_acc16_3']},
}

umbral6  = LIMITE_6G  * UMBRAL_SAT   # 52.97 m/s²
umbral16 = LIMITE_16G * UMBRAL_SAT   # 141.26 m/s²

# ══════════════════════════════════════════════════════════════════════════════
# PASO 1: DETECCIÓN DE SATURACIÓN POR SENSOR ANATÓMICO Y ACTIVIDAD
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 65)
print("  PASO 1: DETECCIÓN DE SATURACIÓN")
print("=" * 65)

registros = []

for ubicacion, cols in SENSORES.items():
    cols6  = [c for c in cols['acc6']  if c in df.columns]
    cols16 = [c for c in cols['acc16'] if c in df.columns]

    if not cols6 or not cols16:
        print(f"  {ubicacion}: columnas no encontradas, omitiendo.")
        continue

    # Máximo valor absoluto en cualquier eje acc6 o acc16 por fila
    df[f'{ubicacion}_max_abs_6g']  = df[cols6].abs().max(axis=1)
    df[f'{ubicacion}_max_abs_16g'] = df[cols16].abs().max(axis=1)

    # Flags de saturación por fila
    df[f'{ubicacion}_sat_6g']  = df[f'{ubicacion}_max_abs_6g']  > umbral6
    df[f'{ubicacion}_sat_16g'] = df[f'{ubicacion}_max_abs_16g'] > umbral16
    # Saturación real: el ±6g está al límite pero el ±16g todavía tiene margen
    df[f'{ubicacion}_sat_real'] = df[f'{ubicacion}_sat_6g'] & ~df[f'{ubicacion}_sat_16g']

    # Resumen por actividad
    for act_id, grupo in df.groupby('activityID'):
        n_total = len(grupo)
        n_sat6  = grupo[f'{ubicacion}_sat_6g'].sum()
        n_sat16 = grupo[f'{ubicacion}_sat_16g'].sum()
        n_real  = grupo[f'{ubicacion}_sat_real'].sum()
        registros.append({
            'ubicacion':  ubicacion,
            'activityID': act_id,
            'nombre':     ACTIVIDADES.get(act_id, str(act_id)),
            'intensidad': INTENSIDAD.get(act_id, 'media'),
            'total':      n_total,
            'n_sat_6g':   n_sat6,
            'n_sat_16g':  n_sat16,
            'n_sat_real': n_real,
            'pct_sat_6g':  n_sat6  / n_total * 100,
            'pct_sat_16g': n_sat16 / n_total * 100,
            'pct_sat_real': n_real / n_total * 100,
        })

    total_global = df[f'{ubicacion}_sat_real'].sum()
    print(f"  {ubicacion.upper():6s} — saturación real acumulada: "
          f"{total_global:,} filas ({total_global/len(df)*100:.2f}%)")

resumen = pd.DataFrame(registros)

# ══════════════════════════════════════════════════════════════════════════════
# PASO 2: VISUALIZACIÓN — 3 PANELES (uno por sensor anatómico)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("  PASO 2: VISUALIZACIONES POR SENSOR ANATÓMICO")
print("=" * 65)

fig, axes = plt.subplots(3, 3, figsize=(20, 18))
fig.suptitle('Análisis de saturación ±6g vs ±16g por sensor anatómico y actividad\n'
             '(datos crudos en m/s² — pre-interpolación)',
             fontsize=14, fontweight='bold', y=1.01)

titulos_col = ['% saturación ±6g', 'Comparación ±6g vs ±16g', 'Saturación real (solo ±6g)']
colores_sensor = {'hand': '#457b9d', 'chest': '#2d6a4f', 'ankle': '#7d3c98'}

for fila, ubicacion in enumerate(['hand', 'chest', 'ankle']):
    sub = resumen[resumen['ubicacion'] == ubicacion].sort_values('pct_sat_6g', ascending=True)
    colores = [COLOR_INT.get(sub['intensidad'].iloc[i], '#aaa') for i in range(len(sub))]

    # ── Col 0: barras % saturación ±6g ────────────────────────────────────────
    ax = axes[fila][0]
    bars = ax.barh(sub['nombre'], sub['pct_sat_6g'], color=colores, edgecolor='white', linewidth=0.5)
    ax.axvline(x=UMBRAL_PCT_DECISION, color='black', linestyle='--',
               linewidth=1, alpha=0.6, label=f'Umbral decisión {UMBRAL_PCT_DECISION}%')
    for bar, val in zip(bars, sub['pct_sat_6g']):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8)
    ax.set_title(f'{ubicacion.upper()} — {titulos_col[0]}', fontsize=10, fontweight='bold',
                 color=colores_sensor[ubicacion])
    ax.set_xlabel('% muestras', fontsize=9)
    if fila == 0:
        leyenda = [Patch(color='#e63946', label='Alta intensidad'),
                   Patch(color='#f4a261', label='Media intensidad'),
                   Patch(color='#2a9d8f', label='Baja intensidad'),
                   Line2D([0],[0], color='black', linestyle='--', label=f'Umbral {UMBRAL_PCT_DECISION}%')]
        ax.legend(handles=leyenda, fontsize=7, loc='lower right')

    # ── Col 1: comparación ±6g vs ±16g lado a lado ────────────────────────────
    ax = axes[fila][1]
    sub_ord = sub.sort_values('pct_sat_6g', ascending=False)
    x = np.arange(len(sub_ord))
    ancho = 0.35
    ax.bar(x - ancho/2, sub_ord['pct_sat_6g'],  ancho, label='±6g',  color='#e63946', alpha=0.85)
    ax.bar(x + ancho/2, sub_ord['pct_sat_16g'], ancho, label='±16g', color='#457b9d', alpha=0.85)
    ax.axhline(y=UMBRAL_PCT_DECISION, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(sub_ord['nombre'], rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('% muestras', fontsize=9)
    ax.set_title(f'{ubicacion.upper()} — {titulos_col[1]}', fontsize=10, fontweight='bold',
                 color=colores_sensor[ubicacion])
    if fila == 0:
        ax.legend(fontsize=8)

    # ── Col 2: saturación real (6g sin 16g) ───────────────────────────────────
    ax = axes[fila][2]
    sub_ord2 = sub.sort_values('pct_sat_real', ascending=False)
    colores2 = [COLOR_INT.get(sub_ord2['intensidad'].iloc[i], '#aaa') for i in range(len(sub_ord2))]
    bars2 = ax.bar(sub_ord2['nombre'], sub_ord2['pct_sat_real'], color=colores2, edgecolor='white', linewidth=0.5)
    ax.axhline(y=UMBRAL_PCT_DECISION, color='black', linestyle='--', linewidth=1, alpha=0.5,
               label=f'Umbral decisión {UMBRAL_PCT_DECISION}%')
    ax.set_xticks(range(len(sub_ord2)))
    ax.set_xticklabels(sub_ord2['nombre'], rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('% muestras', fontsize=9)
    ax.set_title(f'{ubicacion.upper()} — {titulos_col[2]}', fontsize=10, fontweight='bold',
                 color=colores_sensor[ubicacion])

    # Marcar con asterisco las actividades que superan el umbral de decisión
    for bar, val, nombre in zip(bars2, sub_ord2['pct_sat_real'], sub_ord2['nombre']):
        if val > UMBRAL_PCT_DECISION:
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.3,
                    '* reemplazar', ha='center', fontsize=7, color='#a32d2d', fontweight='bold')

plt.tight_layout()
plt.savefig('saturacion_por_sensor_anatomico.png', dpi=150, bbox_inches='tight')
plt.show()
print("  Guardado: saturacion_por_sensor_anatomico.png")

# ══════════════════════════════════════════════════════════════════════════════
# PASO 3: DECISIÓN Y REEMPLAZO SELECTIVO acc6 → acc16 (reescalado)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("  PASO 3: REEMPLAZO SELECTIVO POR ACTIVIDAD Y SENSOR")
print("=" * 65)
print(f"  Umbral de decisión: si una actividad tiene >{UMBRAL_PCT_DECISION}% de")
print(f"  saturación real en una ubicación, se reemplaza acc6 por acc16")
print(f"  reescalado a rango ±6g para mantener unidades comparables.\n")

# Factor de reescalado: convierte acc16 (rango ±157) al rango efectivo de acc6 (±59)
# Esto preserva la proporcionalidad — no clampea, solo escala linealmente.
FACTOR_REESCALADO = LIMITE_6G / LIMITE_16G  # ≈ 0.375

reemplazos_realizados = []

for ubicacion, cols in SENSORES.items():
    cols6  = [c for c in cols['acc6']  if c in df.columns]
    cols16 = [c for c in cols['acc16'] if c in df.columns]

    if not cols6 or not cols16:
        continue

    sub = resumen[(resumen['ubicacion'] == ubicacion)]

    for _, fila_res in sub.iterrows():
        act_id  = fila_res['activityID']
        pct_real = fila_res['pct_sat_real']

        if pct_real > UMBRAL_PCT_DECISION:
            # Máscara: filas de esta actividad donde el ±6g está en zona de saturación
            mascara = (df['activityID'] == act_id) & df[f'{ubicacion}_sat_real']
            n_filas = mascara.sum()

            for c6, c16 in zip(cols6, cols16):
                # Columna flag antes de reemplazar
                flag_col = f'{c6}_sat_flag'
                if flag_col not in df.columns:
                    df[flag_col] = False
                df.loc[mascara, flag_col] = True

                # Reemplazo: acc16 reescalado al rango de acc6
                df.loc[mascara, c6] = df.loc[mascara, c16] * FACTOR_REESCALADO

            reemplazos_realizados.append({
                'ubicacion':  ubicacion,
                'actividad':  fila_res['nombre'],
                'activityID': act_id,
                'pct_sat_real': pct_real,
                'filas_reemplazadas': n_filas,
            })
            print(f"  REEMPLAZO: {ubicacion.upper():6s} | {fila_res['nombre']:22s} "
                  f"| {pct_real:5.1f}% sat | {n_filas:,} filas")
        else:
            print(f"  CONSERVAR: {ubicacion.upper():6s} | {fila_res['nombre']:22s} "
                  f"| {pct_real:5.1f}% sat — acc6 tiene resolución superior")

# ══════════════════════════════════════════════════════════════════════════════
# PASO 4: VISUALIZACIÓN DE DECISIONES + VALIDACIÓN
# ══════════════════════════════════════════════════════════════════════════════

if reemplazos_realizados:
    df_rep = pd.DataFrame(reemplazos_realizados)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Resultado del reemplazo selectivo acc6 → acc16 reescalado',
                 fontsize=13, fontweight='bold')

    # Mapa de calor de decisiones (actividad × ubicación)
    pivot = resumen.pivot_table(index='nombre', columns='ubicacion',
                                values='pct_sat_real', aggfunc='mean').fillna(0)
    im = axes[0].imshow(pivot.values, aspect='auto', cmap='RdYlGn_r',
                        vmin=0, vmax=max(30, resumen['pct_sat_real'].max()))
    axes[0].set_xticks(range(len(pivot.columns)))
    axes[0].set_xticklabels([c.upper() for c in pivot.columns], fontsize=10)
    axes[0].set_yticks(range(len(pivot.index)))
    axes[0].set_yticklabels(pivot.index, fontsize=9)
    plt.colorbar(im, ax=axes[0], label='% saturación real')

    # Marcar celdas con decisión de reemplazo
    for i, act in enumerate(pivot.index):
        for j, ubi in enumerate(pivot.columns):
            val = pivot.values[i, j]
            simbolo = '✕ reemplazar' if val > UMBRAL_PCT_DECISION else ''
            color_txt = 'white' if val > UMBRAL_PCT_DECISION else 'black'
            axes[0].text(j, i, f'{val:.1f}%\n{simbolo}',
                        ha='center', va='center', fontsize=7,
                        color=color_txt, fontweight='bold' if val > UMBRAL_PCT_DECISION else 'normal')
    axes[0].set_title(f'Mapa de decisión (umbral >{UMBRAL_PCT_DECISION}% = reemplazar)',
                      fontsize=11)
    axes[0].axhline(y=-0.5, color='gray', linewidth=0.5)

    # Barras de filas reemplazadas por caso
    etiquetas = [f"{r['ubicacion'].upper()}\n{r['actividad']}" for _, r in df_rep.iterrows()]
    colores_rep = ['#e63946'] * len(df_rep)
    axes[1].barh(etiquetas, df_rep['filas_reemplazadas'], color=colores_rep, edgecolor='white')
    for i, (_, r) in enumerate(df_rep.iterrows()):
        axes[1].text(r['filas_reemplazadas'] + 100, i,
                     f"{r['filas_reemplazadas']:,} filas ({r['pct_sat_real']:.1f}%)",
                     va='center', fontsize=8)
    axes[1].set_xlabel('Filas reemplazadas')
    axes[1].set_title('Filas donde acc6 fue reemplazado por acc16 reescalado', fontsize=11)

    plt.tight_layout()
    plt.savefig('reemplazo_selectivo_resultados.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n  Guardado: reemplazo_selectivo_resultados.png")
else:
    print("\n  No se realizó ningún reemplazo — la saturación está bajo el umbral en todos los casos.")

# ══════════════════════════════════════════════════════════════════════════════
# PASO 5: LIMPIEZA DE COLUMNAS AUXILIARES
# ══════════════════════════════════════════════════════════════════════════════

cols_aux = [c for c in df.columns if
            any(x in c for x in ['_sat_6g', '_sat_16g', '_sat_real', '_max_abs'])]
df = df.drop(columns=cols_aux)

total_flags = sum(1 for c in df.columns if '_sat_flag' in c)
total_filas_flag = df[[c for c in df.columns if '_sat_flag' in c]].any(axis=1).sum() if total_flags > 0 else 0

print("\n" + "=" * 65)
print("  RESUMEN FINAL")
print("=" * 65)
print(f"  Columnas auxiliares de análisis eliminadas")
print(f"  Columnas flag de trazabilidad conservadas : {total_flags}")
print(f"  Filas con al menos un reemplazo realizado : {total_filas_flag:,}")
print(f"  Factor de reescalado aplicado             : {FACTOR_REESCALADO:.4f}")
print(f"  (acc16 × {FACTOR_REESCALADO:.4f} → rango equivalente a acc6)")
print(f"\n  df listo para la siguiente etapa: interpolación por sujeto.")

## 3. Identificacion y limpieza de datos ausentes (interpolacion por sujeto)


In [ ]:
# INTERPOLACION POR SUJETO — Correccion critica
#
# ERROR DEL CODIGO ANTERIOR:
#   df['heart_rate'].interpolate(...)  <-- interpola sobre TODO el df
#   Cuando el sujeto 101 termina y el 102 empieza con NaN, la interpolacion
#   cruza la frontera y rellena con frecuencias cardiacas del sujeto anterior.
#   Eso es contaminacion inter-sujeto y viola la independencia estadistica.
#
# SOLUCION: groupby('subject_id') antes de interpolar.
#   Cada sujeto se trata como una serie temporal completamente independiente.

imu_cols = [col for col in df.columns
            if col not in ['timestamp', 'activityID', 'heart_rate', 'subject_id']]

# Sensores IMU: gaps pequeños (<0.5% NaN) por data dropping inalambrico
# Limite 10 muestras = 0.1 segundos a 100 Hz, suficiente para cubrir los gaps
df[imu_cols] = (
    df.groupby('subject_id')[imu_cols]
    .transform(lambda x: x.interpolate(method='linear', limit=10))
)

# Heart rate: sensor de ~9Hz frente a 100Hz de los IMUs
# El 90% de NaN NO es perdida de datos, es diferencia de frecuencia de muestreo
# Limite 100 muestras = ~1 segundo de ventana de interpolacion cardiaca
# groupby CRITICO: sin esto, el HR del ultimo registro del sujeto 101
# se usaria para interpolar el inicio del sujeto 102
df['heart_rate'] = (
    df.groupby('subject_id')['heart_rate']
    .transform(lambda x: x.interpolate(method='linear', limit=100))
)

# Rellenar extremos residuales (inicio/fin de sesion de cada sujeto)
for col in imu_cols + ['heart_rate']:
    df[col] = df.groupby('subject_id')[col].transform(lambda x: x.ffill().bfill())

nan_restantes = df.isnull().sum().sum()
print(f"Filas tras limpieza   : {df.shape[0]:,}")
print(f"Columnas              : {df.shape[1]:,}")
print(f"Sujetos               : {df['subject_id'].nunique()}")
print(f"NaN restantes         : {nan_restantes}")
print()
print("Filas por sujeto (post-limpieza):")
print(df.groupby('subject_id')['activityID'].count().to_string())


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VERIFICACIÓN DE COHERENCIA FISIOLÓGICA DEL HEART RATE INTERPOLADO
# Y ALTERNATIVAS DE IMPUTACIÓN ROBUSTA
# ═══════════════════════════════════════════════════════════════════════════════
#
# CONTEXTO DEL PROBLEMA:
# La cohorte PAMAP2 tiene media de edad 27.22 ± 3.31 años (adultos jóvenes).
# Los límites fisiológicos de frecuencia cardíaca para este grupo son:
#
#   Reposo absoluto (sueño):     ~40–50 BPM
#   Reposo despierto (Lying):    ~55–75 BPM
#   Actividad moderada:          ~80–140 BPM
#   Alta intensidad (Running):   ~150–185 BPM
#   Máximo teórico (220 - edad): 220 - 27 ≈ 193 BPM
#   Máximo absoluto de seguridad: ~200 BPM
#
# RIESGO DEL ffill/bfill en extremos:
#   Si el sujeto tiene NaN al inicio de su sesión, bfill() jalará el primer
#   valor real (que puede ser 175 BPM de Running) y lo propagará hacia atrás,
#   asignando 175 BPM a los frames de Lying del inicio de la sesión.
#   Esto crea datos fisiológicamente imposibles.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Límites fisiológicos para adultos jóvenes (~27 años) ─────────────────────
HR_MIN_FISIOLOGICO = 40    # BPM — reposo absoluto (umbral de bradicardia clínica)
HR_MAX_FISIOLOGICO = 200   # BPM — máximo absoluto de seguridad para 27 años
HR_REPOSO_TIPICO   = 65    # BPM — valor de imputación para extremos sin contexto


# ══════════════════════════════════════════════════════════════════════════════
# PASO 1: DIAGNÓSTICO DEL ESTADO ACTUAL (antes de cualquier corrección)
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 65)
print("  DIAGNÓSTICO DE HEART RATE — ESTADO ACTUAL")
print("=" * 65)

# 1a. Distribución global de heart rate post-interpolación
hr_valido = df['heart_rate'].dropna()
print(f"\n  Valores de heart_rate presentes: {len(hr_valido):,}")
print(f"  Min          : {hr_valido.min():.1f} BPM")
print(f"  Max          : {hr_valido.max():.1f} BPM")
print(f"  Media        : {hr_valido.mean():.1f} BPM")
print(f"  Mediana      : {hr_valido.median():.1f} BPM")
print(f"  Desv. estándar: {hr_valido.std():.1f} BPM")

# 1b. Detección de valores fuera de rango fisiológico
fuera_min = (df['heart_rate'] < HR_MIN_FISIOLOGICO).sum()
fuera_max = (df['heart_rate'] > HR_MAX_FISIOLOGICO).sum()
nan_restantes = df['heart_rate'].isna().sum()

print(f"\n  Valores < {HR_MIN_FISIOLOGICO} BPM (bradicardia extrema): {fuera_min:,}")
print(f"  Valores > {HR_MAX_FISIOLOGICO} BPM (taquicardia extrema): {fuera_max:,}")
print(f"  NaN restantes                              : {nan_restantes:,}")

# 1c. Análisis por extremos de sujeto: ¿qué pasa en los primeros/últimos frames?
print(f"\n  Análisis de extremos por sujeto (primeros y últimos 200 frames):")
problemas_extremos = []
for sid, grupo in df.groupby('subject_id'):
    inicio = grupo.head(200)['heart_rate']
    final  = grupo.tail(200)['heart_rate']
    nan_ini = inicio.isna().sum()
    nan_fin = final.isna().sum()
    # Verificar si hay valores atípicos en los extremos que sugieran bfill incorrecto
    out_ini = ((inicio < HR_MIN_FISIOLOGICO) | (inicio > HR_MAX_FISIOLOGICO)).sum()
    out_fin = ((final  < HR_MIN_FISIOLOGICO) | (final  > HR_MAX_FISIOLOGICO)).sum()
    if nan_ini > 0 or nan_fin > 0 or out_ini > 0 or out_fin > 0:
        problemas_extremos.append({
            'sujeto': sid, 'nan_inicio': nan_ini, 'nan_final': nan_fin,
            'atipicos_inicio': out_ini, 'atipicos_final': out_fin
        })
        print(f"    Sujeto {sid}: NaN inicio={nan_ini}, NaN final={nan_fin}, "
              f"Atípicos inicio={out_ini}, Atípicos final={out_fin}")

if not problemas_extremos:
    print("    Sin problemas detectados en extremos.")

# 1d. Heart rate por actividad — verificar coherencia fisiológica
print(f"\n  Heart rate medio por actividad (coherencia fisiológica):")
ACTIVIDADES = {
    1: 'Lying', 2: 'Sitting', 3: 'Standing', 4: 'Walking',
    5: 'Running', 6: 'Cycling', 7: 'Nordic Walking',
    12: 'Ascending Stairs', 13: 'Descending Stairs',
    16: 'Vacuum Cleaning', 17: 'Ironing', 24: 'Rope Jumping'
}
# Orden fisiológico esperado: Lying < Sitting < Standing < Walking < ... < Running
resumen_act = df.groupby('activityID')['heart_rate'].agg(['mean','std','min','max']).round(1)
resumen_act.index = [ACTIVIDADES.get(i, str(i)) for i in resumen_act.index]
resumen_act.columns = ['Media BPM', 'Desv.Std', 'Mín', 'Máx']
print(resumen_act.sort_values('Media BPM').to_string())


# ══════════════════════════════════════════════════════════════════════════════
# PASO 2: VISUALIZACIÓN DE DIAGNÓSTICO
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Verificación de coherencia fisiológica — Heart Rate post-interpolación',
             fontsize=13, fontweight='bold')

# Panel 1: Histograma global con límites fisiológicos
ax = axes[0][0]
df['heart_rate'].hist(bins=80, ax=ax, color='#457b9d', alpha=0.8, edgecolor='none')
ax.axvline(x=HR_MIN_FISIOLOGICO, color='red',    linestyle='--', linewidth=1.5,
           label=f'Mín fisiológico ({HR_MIN_FISIOLOGICO} BPM)')
ax.axvline(x=HR_MAX_FISIOLOGICO, color='red',    linestyle='--', linewidth=1.5,
           label=f'Máx fisiológico ({HR_MAX_FISIOLOGICO} BPM)')
ax.axvline(x=HR_REPOSO_TIPICO,   color='green',  linestyle=':',  linewidth=1.5,
           label=f'Reposo típico ({HR_REPOSO_TIPICO} BPM)')
ax.set_xlabel('Heart Rate (BPM)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución global post-interpolación')
ax.legend(fontsize=8)

# Panel 2: Boxplot por sujeto
ax = axes[0][1]
datos_por_sujeto = [df[df['subject_id'] == sid]['heart_rate'].dropna().values
                    for sid in sorted(df['subject_id'].unique())]
ax.boxplot(datos_por_sujeto, labels=[f'S{i}' for i in sorted(df['subject_id'].unique())])
ax.axhline(y=HR_MIN_FISIOLOGICO, color='red',   linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(y=HR_MAX_FISIOLOGICO, color='red',   linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Sujeto')
ax.set_ylabel('BPM')
ax.set_title('Distribución por sujeto')

# Panel 3: HR medio por actividad (orden fisiológico esperado)
ax = axes[0][2]
hr_act = df.groupby('activityID')['heart_rate'].mean().sort_values()
hr_act.index = [ACTIVIDADES.get(i, str(i)) for i in hr_act.index]
hr_act.plot(kind='barh', ax=ax, color='#2a9d8f', alpha=0.85)
ax.axvline(x=HR_MIN_FISIOLOGICO, color='red', linestyle='--', linewidth=1)
ax.axvline(x=HR_MAX_FISIOLOGICO, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('BPM promedio')
ax.set_title('HR medio por actividad\n(debe respetar orden de intensidad)')

# Panel 4: Serie temporal de un sujeto (extremos)
ax = axes[1][0]
sid_muestra = sorted(df['subject_id'].unique())[0]
datos_sid = df[df['subject_id'] == sid_muestra]['heart_rate'].reset_index(drop=True)
# Primeros y últimos 1000 frames
n_show = 1000
ax.plot(range(n_show), datos_sid.head(n_show), color='#457b9d', linewidth=0.8,
        label='Primeros 1000 frames')
ax.plot(range(len(datos_sid)-n_show, len(datos_sid)),
        datos_sid.tail(n_show), color='#e63946', linewidth=0.8, label='Últimos 1000 frames')
ax.axhline(y=HR_MIN_FISIOLOGICO, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.axhline(y=HR_MAX_FISIOLOGICO, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Frame')
ax.set_ylabel('BPM')
ax.set_title(f'Sujeto {sid_muestra}: extremos de sesión')
ax.legend(fontsize=8)

# Panel 5: Valores fuera de rango por sujeto
ax = axes[1][1]
fuera_por_sujeto = df.groupby('subject_id').apply(
    lambda g: ((g['heart_rate'] < HR_MIN_FISIOLOGICO) |
               (g['heart_rate'] > HR_MAX_FISIOLOGICO)).sum()
)
fuera_por_sujeto.plot(kind='bar', ax=ax, color='#e63946', alpha=0.85)
ax.set_xlabel('Sujeto')
ax.set_ylabel('Cantidad de filas')
ax.set_title(f'Valores fuera de rango\n(<{HR_MIN_FISIOLOGICO} o >{HR_MAX_FISIOLOGICO} BPM) por sujeto')
ax.tick_params(axis='x', rotation=0)

# Panel 6: NaN residuales por sujeto
ax = axes[1][2]
nan_por_sujeto = df.groupby('subject_id')['heart_rate'].apply(lambda g: g.isna().sum())
nan_por_sujeto.plot(kind='bar', ax=ax, color='#f4a261', alpha=0.85)
ax.set_xlabel('Sujeto')
ax.set_ylabel('NaN residuales')
ax.set_title('NaN residuales en heart_rate por sujeto\n(0 = interpolación completa)')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('verificacion_heart_rate.png', dpi=150, bbox_inches='tight')
plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# PASO 3: CORRECCIÓN — INTERPOLACIÓN CON CLIP FISIOLÓGICO EN EXTREMOS
# ══════════════════════════════════════════════════════════════════════════════
#
# ESTRATEGIA:
#
# PROBLEMA DEL ffill/bfill SIMPLE:
#   Si el inicio de la sesión de un sujeto tiene NaN, bfill() propaga el primer
#   valor real hacia atrás. Ese primer valor puede ser de alta intensidad
#   (ej. 175 BPM si Running fue la primera actividad registrada), generando
#   frames de Lying con 175 BPM — fisiológicamente imposible.
#
# SOLUCIÓN: Imputación contextual por actividad + clip de seguridad
#   1. Interpolación lineal por sujeto (ya existente, correcta para gaps interiores)
#   2. Para NaN residuales en extremos: imputar con la MEDIANA de heart_rate
#      de esa misma actividad en ese mismo sujeto (si existe), o la mediana
#      global de la actividad como fallback.
#   3. Clip final: forzar que todo valor quede en [HR_MIN_FISIOLOGICO, HR_MAX_FISIOLOGICO]
#
# Esta estrategia es más conservadora y fisiológicamente justificable que
# propagar valores arbitrarios desde extremos no controlados.

print("\n" + "=" * 65)
print("  APLICANDO CORRECCIÓN: imputación contextual + clip fisiológico")
print("=" * 65)

# Calcular medianas de HR por actividad (referencia global, usada como fallback)
mediana_por_actividad = df.groupby('activityID')['heart_rate'].median()

def imputar_hr_extremos(grupo):
    """
    Para un grupo (sujeto), imputa los NaN residuales en heart_rate
    usando la mediana de HR de la misma actividad dentro del mismo sujeto.
    Si no hay datos de esa actividad en el sujeto, usa la mediana global.
    """
    # Mediana por actividad dentro del sujeto
    mediana_local = grupo.groupby('activityID')['heart_rate'].transform('median')
    # Rellenar NaN con mediana local, luego fallback a mediana global
    hr_corregido = grupo['heart_rate'].copy()
    mask_nan = hr_corregido.isna()
    hr_corregido[mask_nan] = mediana_local[mask_nan]
    # Segundo fallback: mediana global por actividad
    mask_nan2 = hr_corregido.isna()
    if mask_nan2.any():
        for act_id in grupo.loc[mask_nan2, 'activityID'].unique():
            m = mediana_por_actividad.get(act_id, HR_REPOSO_TIPICO)
            hr_corregido[mask_nan2 & (grupo['activityID'] == act_id)] = m
    # Clip fisiológico de seguridad
    hr_corregido = hr_corregido.clip(lower=HR_MIN_FISIOLOGICO, upper=HR_MAX_FISIOLOGICO)
    return hr_corregido

# Aplicar la corrección
df['heart_rate'] = df.groupby('subject_id', group_keys=False).apply(imputar_hr_extremos)

# ── Verificación post-corrección ──────────────────────────────────────────────
print("\n  Resultado post-corrección:")
print(f"  NaN restantes          : {df['heart_rate'].isna().sum():,}")
print(f"  Min                    : {df['heart_rate'].min():.1f} BPM")
print(f"  Max                    : {df['heart_rate'].max():.1f} BPM")
print(f"  Media                  : {df['heart_rate'].mean():.1f} BPM")
fuera_min_post = (df['heart_rate'] < HR_MIN_FISIOLOGICO).sum()
fuera_max_post = (df['heart_rate'] > HR_MAX_FISIOLOGICO).sum()
print(f"  Valores < {HR_MIN_FISIOLOGICO} BPM        : {fuera_min_post:,}")
print(f"  Valores > {HR_MAX_FISIOLOGICO} BPM        : {fuera_max_post:,}")

# ── Verificación de coherencia fisiológica post-corrección ────────────────────
print(f"\n  Heart rate medio por actividad (post-corrección):")
resumen_post = df.groupby('activityID')['heart_rate'].agg(['mean','min','max']).round(1)
resumen_post.index = [ACTIVIDADES.get(i, str(i)) for i in resumen_post.index]
resumen_post.columns = ['Media BPM', 'Mín', 'Máx']
print(resumen_post.sort_values('Media BPM').to_string())

print(f"\n  Verificación de orden fisiológico:")
print(f"  Lying ≤ Sitting ≤ Standing ≤ Walking ≤ ... ≤ Running?")
hr_por_act_post = df.groupby('activityID')['heart_rate'].mean()
hr_por_act_post.index = [ACTIVIDADES.get(i, str(i)) for i in hr_por_act_post.index]
orden_ok = (hr_por_act_post.get('Lying', 0)    <= hr_por_act_post.get('Running', 999) and
            hr_por_act_post.get('Sitting', 0)  <= hr_por_act_post.get('Walking', 999) and
            hr_por_act_post.get('Standing', 0) <= hr_por_act_post.get('Running', 999))
print(f"  {'OK — orden fisiológico respetado' if orden_ok else 'REVISAR — orden inconsistente'}")

print("\n" + "=" * 65)
print("  RESUMEN DE ESTRATEGIA APLICADA")
print("=" * 65)
print("""
  1. Interpolación lineal por sujeto (limit=100 para HR, limit=10 para IMU)
     → Cubre los gaps interiores de la sesión (diferencia de frecuencias 9/100 Hz)

  2. Imputación contextual para NaN residuales en extremos
     → Usa la mediana de HR de la misma actividad en el mismo sujeto
     → Fallback: mediana global de la actividad
     → Evita propagar valores de alta intensidad a frames de reposo

  3. Clip fisiológico [40, 200] BPM como salvaguarda final
     → Garantiza que ningún valor interpolado o imputado exceda los límites
       fisiológicos para adultos jóvenes de ~27 años
     → No modifica valores reales, solo corrige artefactos de imputación

  Referencia de límites:
    Máximo teórico (220 - 27 años) = 193 BPM
    Máximo de seguridad aplicado   = 200 BPM (margen de ±3 años de la cohorte)
    Mínimo fisiológico             = 40 BPM  (umbral de bradicardia clínica)
""")

## Tipo de datos por cada atributo


In [ ]:

tipos = df.dtypes.reset_index()
tipos.columns = ['Atributo', 'Tipo']
tipos['Tipo'] = tipos['Tipo'].astype(str)

fig, ax = plt.subplots(figsize=(6, len(tipos) * 0.35))
ax.axis('off')

tabla = ax.table(
    cellText=tipos.values,
    colLabels=tipos.columns,
    cellLoc='left',
    loc='center',
    colWidths=[0.6, 0.4]
)

tabla.auto_set_font_size(False)
tabla.set_fontsize(9)

# Colorear encabezado
for j in range(2):
    tabla[0, j].set_facecolor('#2c3e50')
    tabla[0, j].set_text_props(color='white', fontweight='bold')

# Colorear filas alternas
for i in range(1, len(tipos) + 1):
    for j in range(2):
        if i % 2 == 0:
            tabla[i, j].set_facecolor('#f2f2f2')

plt.title('Tipos de datos por atributo', fontweight='bold', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('tipos_de_datos.png', dpi=150, bbox_inches='tight')
plt.show()

## Resumen estadistico

In [ ]:

import matplotlib.gridspec as gridspec


# Definir grupos
grupos = {
    'Heart Rate': ['heart_rate'],
    'Hand': [col for col in df.columns if 'hand' in col],
    'Chest': [col for col in df.columns if 'chest' in col],
    'Ankle': [col for col in df.columns if 'ankle' in col],
}

colores_header = {
    'Heart Rate': '#c0392b',
    'Hand':       '#1a5276',
    'Chest':      '#1e8449',
    'Ankle':      '#7d3c98',
}

fig = plt.figure(figsize=(20, len(grupos) * 7))
gs = gridspec.GridSpec(len(grupos), 1, figure=fig, hspace=0.6)

for idx, (grupo, cols) in enumerate(grupos.items()):
    cols_existentes = [c for c in cols if c in df.columns]
    if not cols_existentes:
        continue

    stats = df[cols_existentes].describe().T.round(4).reset_index()
    stats.columns = ['Atributo', 'Count', 'Mean', 'Std', 'Min', '25%', '50%', '75%', 'Max']

    ax = fig.add_subplot(gs[idx])
    ax.axis('off')

    tabla = ax.table(
        cellText=stats.values,
        colLabels=stats.columns,
        cellLoc='center',
        loc='center'
    )

    tabla.auto_set_font_size(False)
    tabla.set_fontsize(9)
    tabla.auto_set_column_width(col=list(range(len(stats.columns))))
    tabla.scale(1, 1.4)

    # Encabezado con color del grupo
    for j in range(len(stats.columns)):
        tabla[0, j].set_facecolor(colores_header[grupo])
        tabla[0, j].set_text_props(color='white', fontweight='bold')

    # Filas alternas
    for i in range(1, len(stats) + 1):
        for j in range(len(stats.columns)):
            tabla[i, j].set_facecolor('#f5f5f5' if i % 2 == 0 else 'white')

    ax.set_title(f'Sensor: {grupo}', fontsize=13, fontweight='bold',
                 color=colores_header[grupo], pad=20, loc='left')

plt.suptitle('Resumen Estadístico por Grupo de Sensores',
             fontsize=16, fontweight='bold', y=1.01)

plt.savefig('resumen_estadistico_grupos.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Guardado como 'resumen_estadistico_grupos.png'")

## Distribución de clases

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Mapeo de actividades
actividades = {
    1: 'Lying', 2: 'Sitting', 3: 'Standing', 4: 'Walking',
    5: 'Running', 6: 'Cycling', 7: 'Nordic Walking',
    9: 'Watching TV', 10: 'Computer Work', 11: 'Car Driving',
    12: 'Ascending Stairs', 13: 'Descending Stairs',
    16: 'Vacuum Cleaning', 17: 'Ironing', 18: 'Folding Laundry',
    19: 'House Cleaning', 20: 'Playing Soccer',
    24: 'Rope Jumping'
}

# Conteo por actividad
conteo = df['activityID'].value_counts().sort_index()
conteo.index = [actividades.get(i, str(i)) for i in conteo.index]
conteo = conteo.sort_values(ascending=True)

# Porcentajes
porcentajes = (conteo / conteo.sum() * 100).round(2)

# Colores
cmap = plt.cm.viridis
colors = [cmap(i / len(conteo)) for i in range(len(conteo))]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Gráfica de barras ---
bars = axes[0].barh(conteo.index, conteo.values, color=colors, edgecolor='white')
for bar, pct in zip(bars, porcentajes.values):
    axes[0].text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
                 f'{pct}%', va='center', fontsize=9)
axes[0].set_xlabel('Número de muestras', fontsize=11)
axes[0].set_title('Distribución de clases\n(número de muestras)',
                   fontsize=13, fontweight='bold')
axes[0].set_xlim(0, conteo.max() * 1.15)
axes[0].grid(axis='x', alpha=0.3)

# --- Gráfica de pie ---
axes[1].pie(conteo.values, labels=conteo.index, colors=colors,
            autopct='%1.1f%%', startangle=140,
            textprops={'fontsize': 8})
axes[1].set_title('Distribución de clases\n(proporción)',
                   fontsize=13, fontweight='bold')

plt.suptitle('Distribución de Actividades - PAMAP2',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('distribucion_clases.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardado como 'distribucion_clases.png'")

##Correlación entre atributos

In [ ]:


grupos = {
    'Hand':  [col for col in df.columns if 'hand' in col],
    'Chest': [col for col in df.columns if 'chest' in col] + ['heart_rate'],
    'Ankle': [col for col in df.columns if 'ankle' in col],
}

colores = {
    'Hand':  'coolwarm',
    'Chest': 'coolwarm',
    'Ankle': 'coolwarm',
}

fig, axes = plt.subplots(1, 3, figsize=(26, 10))

for idx, (grupo, cols) in enumerate(grupos.items()):
    cols_existentes = [c for c in cols if c in df.columns]
    ax = axes[idx]
    corr = df[cols_existentes].corr()

    sns.heatmap(
        corr,
        ax=ax,
        cmap='coolwarm',
        center=0,
        vmin=-1, vmax=1,
        annot=True, fmt='.2f',
        linewidths=0.5,
        square=True,
        annot_kws={'size': 7},
        cbar_kws={'shrink': 0.8}
    )

    ax.set_title(f'Correlación - Sensor {grupo}', fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)

plt.suptitle('Matrices de Correlación por Grupo de Sensores - PAMAP2',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('correlacion_por_grupos.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Guardado como 'correlacion_por_grupos.png'")

## Histograma de atributos

In [ ]:
# Configuración de estilo visual
sns.set_theme(style="whitegrid")

# 8. Histograma de Atributos (Selección Estratégica)
# Seleccionamos desde el índice 2 (heart_rate) hasta el 15 (hand_mag3)
# Recordando que en Python el límite superior del rango no es inclusivo, usamos [2:16]
cols_to_plot = df.select_dtypes(include=[np.number]).columns[2:16]

# Generación del plot
df[cols_to_plot].hist(bins=50, figsize=(16, 12), color='teal', edgecolor='black')

# Ajustes estéticos
plt.suptitle('Distribución de Atributos: Frecuencia Cardíaca y Sensores de la Mano', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

# Grafica de densidad



In [ ]:

# Configuración de estilo visual
sns.set_theme(style="whitegrid")

# 9. Gráfica de Densidad (KDE) condicionada por Actividad
# Elegimos un atributo representativo con alto valor biológico: heart_rate
atributo_estudio = 'heart_rate'

plt.figure(figsize=(14, 8))

# Generamos el KDE plot superponiendo las clases (activityID)
# Usamos common_norm=False para que cada curva se escale de forma independiente y sea visible
sns.kdeplot(
    data=df,
    x=atributo_estudio,
    hue='activityID',
    palette='tab20',
    common_norm=False,
    fill=True,
    linewidth=2,
    alpha=0.3
)

# Ajustes estéticos y etiquetas
plt.title(f'Distribución de Densidad (KDE) de {atributo_estudio} por Actividad', fontsize=16)
plt.xlabel('Frecuencia Cardíaca (BPM)', fontsize=12)
plt.ylabel('Densidad de Probabilidad', fontsize=12)

# Mover la leyenda fuera de la gráfica para no tapar las curvas
plt.legend(title='Activity ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

#Boxplot


In [ ]:

# Configuración de estilo visual
sns.set_theme(style="whitegrid")

# 10. Gráfica de Cajas y Bigotes (Agrupación Estratégica)
# Filtramos inteligentemente solo las columnas de los acelerómetros de 16g
columnas_acc16 = [col for col in df.columns if 'acc16' in col]

plt.figure(figsize=(16, 8))

# Generamos el Boxplot
sns.boxplot(
    data=df[columnas_acc16],
    palette='Set2',
    fliersize=2, # Reducimos el tamaño de los puntos atípicos
    linewidth=1.5
)

# Ajustes estéticos y etiquetas
# NOTA: Agregamos la 'r' antes de las comillas para evitar el SyntaxWarning
plt.title(r'Distribución y Dispersión (Boxplot) de Acelerómetros ($\pm16g$)', fontsize=16)
plt.ylabel('Aceleración ($m/s^2$)', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=11)

plt.tight_layout()
plt.show()

#Matriz de correlacion


In [ ]:
# Configuración de estilo visual
sns.set_theme(style="white")

# 11. Matriz de Correlación (Segmentada por ubicación anatómica)
# Definimos los prefijos de las zonas a analizar.
# Añadimos heart_rate al grupo del pecho por proximidad fisiológica.
zonas = {
    'Hand': [col for col in df.columns if 'hand' in col],
    'Chest': ['heart_rate'] + [col for col in df.columns if 'chest' in col],
    'Ankle': [col for col in df.columns if 'ankle' in col]
}

# Generamos un heatmap para cada zona
for nombre_zona, columnas_zona in zonas.items():
    plt.figure(figsize=(12, 10))

    # Calculamos la matriz de correlación de Pearson
    matriz_corr = df[columnas_zona].corr(method='pearson')

    # Generamos el mapa de calor
    sns.heatmap(
        matriz_corr,
        annot=True,          # Muestra el valor numérico
        fmt='.2f',           # Formato a 2 decimales
        cmap='coolwarm',     # Paleta de colores divergente (rojo=positivo, azul=negativo)
        vmin=-1, vmax=1,     # Límites del coeficiente de Pearson
        linewidths=0.5,
        cbar_kws={"shrink": .8}
    )

    # Ajustes estéticos
    plt.title(f'Matriz de Correlación - Sensor {nombre_zona}', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

#Matriz de dispersion


In [ ]:

# Configuración de estilo visual
sns.set_theme(style="whitegrid")

# 12. Matriz de Dispersión (Pairplot) con Submuestreo Estratificado
# Seleccionamos un subconjunto representativo de atributos (uno de cada tipo/zona)
atributos_clave = [
    'heart_rate',      # Fisiológico
    'hand_acc16_1',    # Aceleración (Mano)
    'chest_gyr1',      # Rotación (Pecho)
    'ankle_mag1'       # Magnetismo (Tobillo)
]

# Aplicamos un submuestreo estratificado: 500 muestras por cada activityID
# Esto reduce ~1.9 millones de filas a unas pocas miles, ideal para graficar
df_muestra = df.groupby('activityID', group_keys=False).apply(lambda x: x.sample(min(len(x), 500)))

# Generamos el Pairplot
# Usamos 'corner=True' para no graficar el espejo superior y ahorrar espacio/tiempo
sns.pairplot(
    df_muestra,
    vars=atributos_clave,
    hue='activityID',
    palette='tab20',
    corner=True,
    plot_kws={'alpha': 0.6, 's': 15} # 'alpha' da transparencia, 's' reduce el tamaño del punto
)

# Ajustes estéticos
plt.suptitle('Matriz de Dispersión Multivariada (Muestra Estratificada)', y=1.02, fontsize=16)
plt.show()

#Estandarizacion


In [ ]:

# 13. Estandarización de la Base de Datos (Z-Score)

# Instanciamos el escalador
scaler = StandardScaler()

# Identificamos las columnas numéricas que representan atributos físicos.
# EXCLUIMOS explícitamente el 'timestamp' y el 'activityID' para no alterar su significado.
columnas_a_escalar = [col for col in df.columns if col not in
                      ['timestamp', 'activityID', 'subject_id']]

# Creamos una copia del dataframe para mantener los datos crudos a salvo por si se requieren después
df_estandarizado = df.copy()

# Aplicamos la transformación matemática
df_estandarizado[columnas_a_escalar] = scaler.fit_transform(df[columnas_a_escalar])

# Comprobación de integridad: Verificamos que las nuevas medias tiendan a 0 y las desviaciones a 1
resumen_escalado = df_estandarizado[columnas_a_escalar].describe().loc[['mean', 'std']].round(3)

print("Verificación de la Estandarización:")
display(resumen_escalado)

## Conversion a float32 para mayor optimización


In [ ]:
# Optimización de memoria antes de extracción de características
memoria_antes = df_estandarizado.memory_usage(deep=True).sum() / 1024**2

cols_sensores = [col for col in df_estandarizado.columns if col not in
                 ['timestamp', 'activityID', 'subject_id']]

df_estandarizado[cols_sensores] = df_estandarizado[cols_sensores].astype('float32')
df_estandarizado['activityID'] = df_estandarizado['activityID'].astype('int8')
df_estandarizado['subject_id'] = df_estandarizado['subject_id'].astype('int8')
df_estandarizado['timestamp'] = df_estandarizado['timestamp'].astype('float32')

memoria_despues = df_estandarizado.memory_usage(deep=True).sum() / 1024**2

print(f"Memoria antes : {memoria_antes:.2f} MB")
print(f"Memoria después: {memoria_despues:.2f} MB")
print(f"Ahorro        : {memoria_antes - memoria_despues:.2f} MB ({100 - (memoria_despues/memoria_antes*100):.1f}%)")

## Extraccion de caracteristicas


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

FRECUENCIA_HZ = 100
VENTANA_SEG = 3
MUESTRAS_POR_VENTANA = FRECUENCIA_HZ * VENTANA_SEG  # 300

print("=" * 60)
print("  INICIANDO EXTRACCIÓN DE CARACTERÍSTICAS")
print("=" * 60)
print(f"  Tamaño de ventana : {VENTANA_SEG} segundos ({MUESTRAS_POR_VENTANA} muestras)")
print(f"  Dataset original  : {df_estandarizado.shape[0]:,} filas crudas")

# ══════════════════════════════════════════════════════════════
# window_id por sujeto para no mezclar datos entre sujetos
# ══════════════════════════════════════════════════════════════
df_estandarizado['window_id'] = (
    df_estandarizado.groupby('subject_id').cumcount() // MUESTRAS_POR_VENTANA
)

# Columnas de sensores agrupadas por ubicación y tipo
# Cada grupo de 3 columnas representa los ejes X, Y, Z de un sensor
grupos_sensores = {
    'heart_rate':    ['heart_rate'],
    'hand_temp':     ['hand_temp'],
    'hand_acc16':    ['hand_acc16_1', 'hand_acc16_2', 'hand_acc16_3'],
    'hand_acc6':     ['hand_acc6_1',  'hand_acc6_2',  'hand_acc6_3'],
    'hand_gyr':      ['hand_gyr1',    'hand_gyr2',    'hand_gyr3'],
    'hand_mag':      ['hand_mag1',    'hand_mag2',    'hand_mag3'],
    'chest_temp':    ['chest_temp'],
    'chest_acc16':   ['chest_acc16_1','chest_acc16_2','chest_acc16_3'],
    'chest_acc6':    ['chest_acc6_1', 'chest_acc6_2', 'chest_acc6_3'],
    'chest_gyr':     ['chest_gyr1',   'chest_gyr2',   'chest_gyr3'],
    'chest_mag':     ['chest_mag1',   'chest_mag2',   'chest_mag3'],
    'ankle_temp':    ['ankle_temp'],
    'ankle_acc16':   ['ankle_acc16_1','ankle_acc16_2','ankle_acc16_3'],
    'ankle_acc6':    ['ankle_acc6_1', 'ankle_acc6_2', 'ankle_acc6_3'],
    'ankle_gyr':     ['ankle_gyr1',   'ankle_gyr2',   'ankle_gyr3'],
    'ankle_mag':     ['ankle_mag1',   'ankle_mag2',   'ankle_mag3'],
}

def extraer_16_features(ventana, nombre_sensor, cols):
    """
    Extrae las 16 características definidas en Garcia-Ceja et al. (2018):
    Para sensores de 3 ejes:
      - Mean eje X, Y, Z           (3 features)
      - Std  eje X, Y, Z           (3 features)
      - Max  eje X, Y, Z           (3 features)
      - Correlación XY, XZ, YZ    (3 features)
      - Magnitud media             (1 feature)
      - Magnitud std               (1 feature)
      - Magnitud AUC               (1 feature)
      - Magnitud diff media        (1 feature)
    Total: 16 features por sensor triaxial

    Para sensores de 1 eje (temperatura, heart_rate):
      - Mean, Std, Max, AUC        (4 features)
    """
    feats = {}

    if len(cols) == 3:
        x = ventana[cols[0]].values
        y = ventana[cols[1]].values
        z = ventana[cols[2]].values

        # 1-3: Media por eje
        feats[f'{nombre_sensor}_mean_x'] = np.mean(x)
        feats[f'{nombre_sensor}_mean_y'] = np.mean(y)
        feats[f'{nombre_sensor}_mean_z'] = np.mean(z)

        # 4-6: Desviación estándar por eje
        feats[f'{nombre_sensor}_std_x'] = np.std(x)
        feats[f'{nombre_sensor}_std_y'] = np.std(y)
        feats[f'{nombre_sensor}_std_z'] = np.std(z)

        # 7-9: Máximo por eje
        feats[f'{nombre_sensor}_max_x'] = np.max(np.abs(x))
        feats[f'{nombre_sensor}_max_y'] = np.max(np.abs(y))
        feats[f'{nombre_sensor}_max_z'] = np.max(np.abs(z))

        # 10-12: Correlación entre pares de ejes
        feats[f'{nombre_sensor}_corr_xy'] = pearsonr(x, y)[0] if np.std(x) > 0 and np.std(y) > 0 else 0
        feats[f'{nombre_sensor}_corr_xz'] = pearsonr(x, z)[0] if np.std(x) > 0 and np.std(z) > 0 else 0
        feats[f'{nombre_sensor}_corr_yz'] = pearsonr(y, z)[0] if np.std(y) > 0 and np.std(z) > 0 else 0

        # Magnitud euclidiana
        magnitud = np.sqrt(x**2 + y**2 + z**2)

        # 13: Magnitud media
        feats[f'{nombre_sensor}_mag_mean'] = np.mean(magnitud)

        # 14: Magnitud std
        feats[f'{nombre_sensor}_mag_std'] = np.std(magnitud)

        # 15: Magnitud AUC (área bajo la curva = suma / N)
        feats[f'{nombre_sensor}_mag_auc'] = np.sum(magnitud) / len(magnitud)

        # 16: Diferencias medias entre muestras consecutivas de magnitud
        feats[f'{nombre_sensor}_mag_diff'] = np.mean(np.abs(np.diff(magnitud)))

    else:
        # Sensor de 1 eje (temperatura, heart_rate)
        v = ventana[cols[0]].values
        feats[f'{nombre_sensor}_mean'] = np.mean(v)
        feats[f'{nombre_sensor}_std']  = np.std(v)
        feats[f'{nombre_sensor}_max']  = np.max(np.abs(v))
        feats[f'{nombre_sensor}_auc']  = np.sum(np.abs(v)) / len(v)

    return feats

# ══════════════════════════════════════════════════════════════
# Extracción de características por ventana y sujeto
# ══════════════════════════════════════════════════════════════
print("  Extrayendo 16 características por sensor...")

registros = []
grupos = df_estandarizado.groupby(['subject_id', 'window_id'])

for (sid, wid), ventana in grupos:
    fila = {'subject_id': sid, 'window_id': wid}

    # Etiqueta por moda
    moda = ventana['activityID'].mode()
    fila['activityID'] = moda[0] if len(moda) > 0 else np.nan

    # Pureza
    fila['pureza'] = ventana['activityID'].value_counts().iloc[0] / len(ventana)

    # Extraer features por grupo de sensor
    for nombre_sensor, cols in grupos_sensores.items():
        cols_presentes = [c for c in cols if c in ventana.columns]
        if cols_presentes:
            feats = extraer_16_features(ventana, nombre_sensor, cols_presentes)
            fila.update(feats)

    registros.append(fila)

df_features = pd.DataFrame(registros)

# ══════════════════════════════════════════════════════════════
# Filtros
# ══════════════════════════════════════════════════════════════
ventanas_antes = len(df_features)
df_features = df_features[df_features['pureza'] > 0.9]
df_features = df_features[df_features['activityID'] != 0]
df_features = df_features.drop(columns=['pureza', 'window_id'])
df_features = df_features.dropna()

# Eliminar columna auxiliar del df original
df_estandarizado.drop(columns=['window_id'], inplace=True)

print("\n" + "=" * 60)
print("  EXTRACCIÓN COMPLETADA CON ÉXITO")
print("=" * 60)
print(f"  Dimensiones del dataset de features : {df_features.shape}")
print(f"  Sujetos                             : {df_features['subject_id'].nunique()}")
print(f"  Ventanas antes de filtros           : {ventanas_antes:,}")
print(f"  Ventanas finales                    : {len(df_features):,}")
print(f"  Features por ventana                : {df_features.shape[1] - 2}")
print(f"\n  Distribución por actividad:")
act_nombres = {1:'Lying',2:'Sitting',3:'Standing',4:'Walking',
               5:'Running',6:'Cycling',7:'Nordic Walking',
               12:'Ascending Stairs',13:'Descending Stairs',
               16:'Vacuum Cleaning',17:'Ironing',24:'Rope Jumping'}
dist = df_features['activityID'].value_counts().sort_index()
dist.index = [act_nombres.get(i, str(i)) for i in dist.index]
print(dist)
display(df_features.head())

## Clasificación

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA, KernelPCA
from sklearn.feature_selection import SelectKBest, f_classif

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ══════════════════════════════════════════════════════════════════
# 1. PREPARACIÓN DE DATOS
# ══════════════════════════════════════════════════════════════════
X = df_features.drop(columns=['activityID', 'subject_id'])
y = df_features['activityID']
groups = df_features['subject_id']

print("🔍 Verificando número de características...")
n_features = X.shape[1]
print(f"Número de features detectadas: {n_features}")
assert n_features == 208, "❌ ERROR: El número de features no coincide con lo esperado (208)"
print("✅ Correcto: Se están utilizando 208 features (16 por sensor)")

ACTIVIDADES = {
    1: 'Lying', 2: 'Sitting', 3: 'Standing', 4: 'Walking',
    5: 'Running', 6: 'Cycling', 7: 'Nordic Walking',
    12: 'Ascending Stairs', 13: 'Descending Stairs',
    16: 'Vacuum Cleaning', 17: 'Ironing', 24: 'Rope Jumping'
}

# ══════════════════════════════════════════════════════════════════
# 2. CLASIFICADORES
# ══════════════════════════════════════════════════════════════════
clasificadores = {
    'Regresión Logística': LogisticRegression(
        max_iter=1000, random_state=42, solver='lbfgs', multi_class='multinomial'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    'Perceptrón': Perceptron(
        max_iter=1000, random_state=42, tol=1e-3
    )
}

# ══════════════════════════════════════════════════════════════════
# 3. TRANSFORMACIONES
# ══════════════════════════════════════════════════════════════════
k_best = max(5, int(n_features * 0.6))  # 124

# ══════════════════════════════════════════════════════════════════
# 4. FUNCIÓN DE EVALUACIÓN LOSO
# ══════════════════════════════════════════════════════════════════
logo = LeaveOneGroupOut()

def evaluar_loso(X_data, y_data, groups, transformacion_fn, nombre_metodo):
    print(f"\n\n{'#'*70}")
    print(f"   MÉTODO: {nombre_metodo}")
    print(f"{'#'*70}")

    # acumuladores por clasificador
    acumuladores = {nombre: {'acc': [], 'y_true': [], 'y_pred': []}
                    for nombre in clasificadores}

    for fold, (train_idx, test_idx) in enumerate(logo.split(X_data, y_data, groups)):
        X_tr, X_ts = X_data.iloc[train_idx], X_data.iloc[test_idx]
        y_tr, y_ts = y_data.iloc[train_idx], y_data.iloc[test_idx]
        subj_test  = groups.iloc[test_idx].unique()[0]

        # Escalar dentro del fold
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_ts_s = scaler.transform(X_ts)

        # Aplicar transformación adicional (PCA, KPCA, Selección)
        X_tr_t, X_ts_t = transformacion_fn(X_tr_s, X_ts_s, y_tr)

        for nombre, modelo in clasificadores.items():
            modelo_fold = type(modelo)(**modelo.get_params())
            modelo_fold.fit(X_tr_t, y_tr)
            y_pred = modelo_fold.predict(X_ts_t)

            acumuladores[nombre]['acc'].append(accuracy_score(y_ts, y_pred))
            acumuladores[nombre]['y_true'].extend(y_ts.tolist())
            acumuladores[nombre]['y_pred'].extend(y_pred.tolist())

    # Resultados por clasificador
    resultados = {}
    print(f"\n{'='*50}")
    for nombre in clasificadores:
        acc_media = np.mean(acumuladores[nombre]['acc'])
        acc_std   = np.std(acumuladores[nombre]['acc'])
        print(f"\n{nombre}")
        print(f"Accuracy LOSO: {acc_media:.4f} ± {acc_std:.4f} ({acc_media*100:.2f}%)")
        print(classification_report(
            acumuladores[nombre]['y_true'],
            acumuladores[nombre]['y_pred'],
            target_names=[ACTIVIDADES[i] for i in sorted(y.unique())]
        ))
        resultados[nombre] = {
            'accuracy': acc_media,
            'acc_std' : acc_std,
            'y_true'  : acumuladores[nombre]['y_true'],
            'y_pred'  : acumuladores[nombre]['y_pred']
        }

    return resultados

# ══════════════════════════════════════════════════════════════════
# 5. DEFINICIÓN DE TRANSFORMACIONES
# ══════════════════════════════════════════════════════════════════
def t_sin_reduccion(Xtr, Xts, ytr):
    return Xtr, Xts

def t_pca(Xtr, Xts, ytr):
    pca = PCA(n_components=0.95)
    return pca.fit_transform(Xtr), pca.transform(Xts)

def t_kpca(Xtr, Xts, ytr):
    kpca = KernelPCA(n_components=min(20, Xtr.shape[1]), kernel='rbf', gamma=0.01)
    return kpca.fit_transform(Xtr), kpca.transform(Xts)

def t_seleccion(Xtr, Xts, ytr):
    selector = SelectKBest(score_func=f_classif, k=k_best)
    return selector.fit_transform(Xtr, ytr), selector.transform(Xts)

transformaciones = {
    'Sin Reducción': t_sin_reduccion,
    'PCA'          : t_pca,
    'Kernel PCA'   : t_kpca,
    'Selección'    : t_seleccion
}

# ══════════════════════════════════════════════════════════════════
# 6. EJECUCIÓN
# ══════════════════════════════════════════════════════════════════
todos_resultados = {}

for nombre_t, fn_t in transformaciones.items():
    todos_resultados[nombre_t] = evaluar_loso(X, y, groups, fn_t, nombre_t)

# ══════════════════════════════════════════════════════════════════
# 7. COMPARACIÓN FINAL
# ══════════════════════════════════════════════════════════════════
print("\n\n" + "="*70)
print("COMPARACIÓN FINAL — LOSO")
print("="*70)
for metodo, res in todos_resultados.items():
    print(f"\n--- {metodo} ---")
    for modelo, datos in res.items():
        print(f"{modelo:20} {datos['accuracy']*100:.2f}% ± {datos['acc_std']*100:.2f}%")

# ══════════════════════════════════════════════════════════════════
# 8. GRÁFICA — MEJOR CONFIGURACIÓN (Sin Reducción)
# ══════════════════════════════════════════════════════════════════
res_base = todos_resultados['Sin Reducción']

df_plot = pd.DataFrame({
    'Modelo'  : list(res_base.keys()),
    'Accuracy': [res_base[m]['accuracy'] for m in res_base]
}).sort_values(by='Accuracy', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(df_plot['Modelo'], df_plot['Accuracy'])

for bar, acc in zip(bars, df_plot['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.002,
            f'{acc*100:.2f}%',
            ha='center', fontweight='bold')

ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Comparación de Clasificadores LOSO (Sin Reducción)')
ax.axhline(y=df_plot['Accuracy'].max(), linestyle='--')

plt.tight_layout()
plt.savefig('comparacion_final_clasificadores_loso.png')
plt.show()
print("✅ Guardado: comparacion_final_clasificadores_loso.png")

# ══════════════════════════════════════════════════════════════════
# 9. MATRIZ DE CONFUSIÓN — MEJOR MODELO GLOBAL
# ══════════════════════════════════════════════════════════════════
# Buscar mejor modelo en todas las configuraciones
mejor_acc  = 0
mejor_clf  = None
mejor_conf = None
mejor_t    = None

for nombre_t, res in todos_resultados.items():
    for clf_nombre, datos in res.items():
        if datos['accuracy'] > mejor_acc:
            mejor_acc  = datos['accuracy']
            mejor_clf  = clf_nombre
            mejor_conf = (datos['y_true'], datos['y_pred'])
            mejor_t    = nombre_t

print(f"\n📌 Mejor modelo: {mejor_clf} con {mejor_t} → {mejor_acc*100:.2f}%")

etiquetas = [ACTIVIDADES[i] for i in sorted(y.unique())]
cm = confusion_matrix(mejor_conf[0], mejor_conf[1], labels=sorted(y.unique()))

fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=etiquetas)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title(f'Matriz de Confusión LOSO — {mejor_clf} ({mejor_t})', fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix_loso.png')
plt.show()
print("✅ Guardado: confusion_matrix_loso.png")

## Validación Cruzada

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

# ══════════════════════════════════════════════════════════════════
# VALIDACIÓN CRUZADA — LOSO (Leave-One-Subject-Out)
# ══════════════════════════════════════════════════════════════════
X_full  = df_features.drop(columns=['activityID', 'subject_id'])
y_full  = df_features['activityID']
groups  = df_features['subject_id']  # ← único cambio estructural

logo = LeaveOneGroupOut()  # 9 folds, 1 sujeto por fold

scoring = {
    'accuracy' : 'accuracy',
    'precision': 'precision_weighted',
    'recall'   : 'recall_weighted',
    'f1'       : 'f1_weighted'
}

# ══════════════════════════════════════════════════════════════════
# CLASIFICADORES
# ══════════════════════════════════════════════════════════════════
clasificadores = {
    'Regresión Logística': LogisticRegression(
        max_iter=1000, random_state=42, solver='lbfgs', multi_class='multinomial'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    'Perceptrón': Perceptron(
        max_iter=1000, random_state=42, tol=1e-3
    )
}

# ══════════════════════════════════════════════════════════════════
# TRANSFORMACIONES
# ══════════════════════════════════════════════════════════════════
n_features = X_full.shape[1]
k_best     = max(5, int(n_features * 0.6))

transformaciones = {
    'Sin Reducción': [('scaler', StandardScaler())],
    'PCA'          : [('scaler', StandardScaler()), ('reduce', PCA(n_components=0.95))],
    'Kernel PCA'   : [('scaler', StandardScaler()), ('reduce', KernelPCA(n_components=min(20, n_features), kernel='rbf', gamma=0.01))],
    'Selección'    : [('scaler', StandardScaler()), ('reduce', SelectKBest(score_func=f_classif, k=k_best))]
}

# ══════════════════════════════════════════════════════════════════
# EJECUCIÓN
# ══════════════════════════════════════════════════════════════════
results = []

for ds_name, pasos in transformaciones.items():
    print(f"\n{'='*60}")
    print(f"  Dataset: {ds_name}")
    print(f"{'='*60}")

    for clf_name, clf in clasificadores.items():
        print(f"  → {clf_name}...", end=" ", flush=True)

        pipeline = Pipeline(pasos + [('clf', clf)])

        scores = cross_validate(
            pipeline, X_full, y_full,
            cv=logo,
            groups=groups,   # ← LOSO usa groups
            scoring=scoring,
            n_jobs=-1
        )

        results.append({
            'Dataset'     : ds_name,
            'Clasificador': clf_name,
            'Accuracy'    : f"{scores['test_accuracy'].mean():.4f} ± {scores['test_accuracy'].std():.4f}",
            'Precision'   : f"{scores['test_precision'].mean():.4f} ± {scores['test_precision'].std():.4f}",
            'Recall'      : f"{scores['test_recall'].mean():.4f} ± {scores['test_recall'].std():.4f}",
            'F1-Score'    : f"{scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}"
        })

        print(f"Accuracy = {scores['test_accuracy'].mean():.4f}")

# ══════════════════════════════════════════════════════════════════
# TABLA FINAL
# ══════════════════════════════════════════════════════════════════
df_cv = pd.DataFrame(results)
print("\n\n" + "="*70)
print("RESULTADOS VALIDACIÓN CRUZADA — LOSO (9 folds, 1 sujeto/fold)")
print("="*70)
print(df_cv.to_string(index=False))

df_cv.to_csv('loso_clasificadores.csv', index=False)
print("\n✅ Guardado: loso_clasificadores.csv")

## Fusion de datos

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier, BaggingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
import pandas as pd
import numpy as np

# ══════════════════════════════════════════════════════════════════
# DATOS
# ══════════════════════════════════════════════════════════════════
X      = df_features.drop(columns=['activityID', 'subject_id'])
y      = df_features['activityID']
groups = df_features['subject_id']  # ← LOSO

logo = LeaveOneGroupOut()  # 9 folds, 1 sujeto por fold

n_features = X.shape[1]  # 208
k_best     = max(5, int(n_features * 0.6))  # 124

scoring = {
    'accuracy' : 'accuracy',
    'precision': 'precision_weighted',
    'recall'   : 'recall_weighted',
    'f1'       : 'f1_weighted'
}

# ══════════════════════════════════════════════════════════════════
# MÉTODOS DE FUSIÓN
# ══════════════════════════════════════════════════════════════════
bagging_rf = BaggingClassifier(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    n_estimators=10,
    random_state=42,
    n_jobs=-1
)

voting = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs', multi_class='multinomial')),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
        ('nb', GaussianNB())
    ],
    voting='soft'
)

adaboost = AdaBoostClassifier(
    n_estimators=100,
    random_state=42,
    algorithm='SAMME'
)

metodos_fusion = {
    'Agregación RF (Bagging)': bagging_rf,
    'Votación (LR + RF + NB)': voting,
    'AdaBoost'               : adaboost
}

# ══════════════════════════════════════════════════════════════════
# TRANSFORMACIONES
# ══════════════════════════════════════════════════════════════════
transformaciones = {
    'Sin Reducción': [('scaler', StandardScaler())],
    'PCA'          : [('scaler', StandardScaler()), ('reduce', PCA(n_components=0.95))],
    'Selección'    : [('scaler', StandardScaler()), ('reduce', SelectKBest(score_func=f_classif, k=k_best))]
}

# ══════════════════════════════════════════════════════════════════
# VALIDACIÓN CRUZADA LOSO
# ══════════════════════════════════════════════════════════════════
results = []

for ds_name, pasos in transformaciones.items():
    print(f"\n{'='*60}")
    print(f"  Dataset: {ds_name}")
    print(f"{'='*60}")

    for fusion_name, fusion_clf in metodos_fusion.items():
        print(f"  → {fusion_name}...", end=" ", flush=True)

        pipeline = Pipeline(pasos + [('clf', fusion_clf)])

        scores = cross_validate(
            pipeline, X, y,
            cv=logo,
            groups=groups,   # ← LOSO usa groups
            scoring=scoring,
            n_jobs=-1
        )

        results.append({
            'Dataset'      : ds_name,
            'Método Fusión': fusion_name,
            'Accuracy'     : f"{scores['test_accuracy'].mean():.4f} ± {scores['test_accuracy'].std():.4f}",
            'Precision'    : f"{scores['test_precision'].mean():.4f} ± {scores['test_precision'].std():.4f}",
            'Recall'       : f"{scores['test_recall'].mean():.4f} ± {scores['test_recall'].std():.4f}",
            'F1-Score'     : f"{scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}"
        })

        print(f"Accuracy = {scores['test_accuracy'].mean():.4f}")

# ══════════════════════════════════════════════════════════════════
# TABLA FINAL
# ══════════════════════════════════════════════════════════════════
df_fusion = pd.DataFrame(results)
print("\n\n" + "="*70)
print("RESULTADOS MÉTODOS DE FUSIÓN — LOSO (9 folds, 1 sujeto/fold)")
print("="*70)
print(df_fusion.to_string(index=False))

df_fusion.to_csv('loso_fusion_results.csv', index=False)
print("\n✅ Guardado: loso_fusion_results.csv")